# Kirby: reversible single-unit pruning

Upload the new kirby_colab.zip and run the three cells in order in the existing Colab GPU workspace. This actually zeros one MLP output column at a time and restores it after measurement. Four activation-ranked units are screened on eight training questions. A qualifying candidate and one seeded random unit are confirmed on the other 56 training questions. Lookup controls are reused. If no candidate passes the screening guards, confirmation is skipped. Maximum 1,736 generation calls; the actual count depends on neutral errors and screening results. No Astra/API key, cumulative edits, retained checkpoint, development or final evaluation.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, re, signal, stat, subprocess, sys, uuid, zipfile

# If several workspaces exist, paste the intended backend path here.
BACKEND_PATH = ""
WORKER_PYTHON = ""
if BACKEND_PATH:
    backend = Path(BACKEND_PATH)
else:
    candidates = sorted(Path("/content").glob("kirby-*/backend"))
    candidates = [p for p in candidates if (p / "data" / "plan-dev-256" / "development.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set BACKEND_PATH to the backend with your 256-question plan. Found: "
                           + str([str(p) for p in candidates])
                           + ". If the runtime was reset, use the main notebook to rebuild the baseline first.")
    backend = candidates[0]
workspace = backend.parent
if not (backend / "data" / "plan-dev-256" / "development.json").is_file():
    raise RuntimeError("This workspace has no 256-question development plan.")
workers = sorted(workspace.glob("worker-venv-*/bin/python"),
                 key=lambda p: p.parent.parent.stat().st_mtime)
worker_python = WORKER_PYTHON or (str(workers[-1]) if workers else "")
if not worker_python or not Path(worker_python).is_file():
    raise RuntimeError("Worker environment missing. Run setup in the main notebook first.")
print("Updating:", backend)
bundle_bytes = globals().pop("KIRBY_BUNDLE_BYTES", None)
if bundle_bytes is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only the newly built kirby_colab.zip.")
    bundle_bytes = next(iter(uploaded.values()))
    del uploaded
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 5_000_000:
        raise ValueError("Unexpected source bundle size.")
    replacements = {}
    for member in members:
        name = PurePosixPath(member.filename)
        allowed = (str(name) in {"backend/README.md", "backend/pyproject.toml"} or
                   (len(name.parts) == 3 and name.parts[0] == "backend" and
                    ((name.parts[1] in {"kirby", "tests"} and name.suffix == ".py") or
                     (name.parts[1] == "notebooks" and name.suffix == ".ipynb") or
                     (name.parts[1] == "docs" and name.suffix == ".md"))))
        if (not allowed or ".." in name.parts or name.is_absolute() or
                "\\" in member.filename or ":" in member.filename or
                stat.S_ISLNK(member.external_attr >> 16) or member.filename in replacements):
            raise ValueError("Unsafe or unexpected source bundle member.")
        replacements[member.filename] = archive.read(member)
    required = {"backend/kirby/pruning.py", "backend/kirby/patches.py", "backend/kirby/evidence.py"}
    if not required.issubset(replacements):
        raise ValueError("Upload the new ZIP containing the pruning experiment.")
for name, payload in replacements.items():
    target = workspace / name
    if target.is_symlink() or not target.resolve().is_relative_to(backend.resolve()):
        raise ValueError("Source destination leaves the backend workspace.")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
del bundle_bytes, replacements

def run_setup(command, **kwargs):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs)
    print(result.stdout, end="")
    result.check_returncode()
    return result

def run_worker(command, *, env=None):
    process = subprocess.Popen(command, cwd=backend, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            remaining, _ = process.communicate(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            remaining, _ = process.communicate()
        print(remaining, end="")
        raise
    finally:
        process.stdout.close()
    if code:
        raise subprocess.CalledProcessError(code, command)

MODEL_ID = "johnsnowlabs/JSL-MedPhi2-2.7B"
MODEL_REVISION = "b313a242eb211863f7b62624b345b321a3b114ea"
PRECISION = "float32"  # Matches your T4 baseline.
MAX_SECONDS = 3600
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HOME", "/content/kirby-hf-cache")
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
revision = MODEL_REVISION
(workspace / "model-revision.txt").write_text(revision, encoding="utf-8")
plan = backend / "data" / "plan-dev-256"
runs = backend / "runs"
run_setup([worker_python, "-c", "import torch; from transformers import PhiConfig, PhiForCausalLM; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))"], cwd=backend)
run_setup([worker_python, "-m", "pip", "install", "-e", str(backend)], cwd=backend)
run_setup([worker_python, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=backend)
saved_plan = json.loads((plan / 'development.json').read_text())
if len(saved_plan['diagnostic_ids']) < 64:
    raise RuntimeError('This pilot needs 64 frozen training diagnostic IDs.')
print('Ready: reversible pruning; four single units, 8 screening / 56 confirmation questions. No Astra or final evaluation.')


In [ ]:
# Re-running this cell shows the same evidence. Set NEW_ATTEMPT=True only to run another pruning.
NEW_ATTEMPT = False
id_file = workspace / 'cue-pruning-v1-64-attempt-id.txt'
if NEW_ATTEMPT or not id_file.is_file():
    id_file.write_text(str(uuid.uuid4()), encoding='utf-8')
attempt_id = str(uuid.UUID(id_file.read_text().strip()))
pruning_report_path = runs / attempt_id / 'report.json'
print('Selected pruning:', attempt_id)
if pruning_report_path.is_file():
    print('This pruning already ran. Showing saved evidence; no new inference.')
elif pruning_report_path.parent.exists():
    raise RuntimeError('This attempt directory exists without a report. Inspect it before selecting NEW_ATTEMPT=True.')
else:
    worker_env = os.environ.copy()
    worker_env.pop('OPENAI_API_KEY', None)
    run_worker([worker_python, '-u', '-m', 'kirby', 'pruning-screen',
                '--manifest', str(plan / 'development.json'), '--attempt-id', attempt_id,
                '--model', MODEL_ID, '--revision', MODEL_REVISION, '--precision', PRECISION,
                '--allow-download', '--questions', '64', '--screening-count', '8', '--candidates', '4', '--max-seconds', str(MAX_SECONDS),
                '--output', str(runs)], env=worker_env)
report = json.loads(pruning_report_path.read_text())
if report.get('protocol') != 'single_unit_cue_pruning_v1':
    raise RuntimeError('Selected report is not pruning v1. Inspect the attempt ID; do not substitute an older result.')
print(json.dumps({key: report.get(key) for key in
                 ('state', 'outcome', 'selected', 'confirmation_signal', 'screening', 'confirmation', 'weights_restored', 'outputs_restored', 'usage', 'error')}, indent=2))
print('Pruning evidence:', pruning_report_path)


In [ ]:
from datetime import datetime, timezone
selected_report = globals().get('pruning_report_path')
if selected_report is None or not Path(selected_report).is_file():
    print('No report for this pruning exists yet. No older experiment will be mislabeled as this result.')
else:
    inventory = subprocess.check_output([worker_python, '-m', 'pip', 'list', '--format=json'], text=True)
    (workspace / 'package-versions.json').write_text(inventory, encoding='utf-8')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    destination = workspace / f'kirby-pruning-evidence-{stamp}.zip'
    subprocess.run([worker_python, '-m', 'kirby.evidence', '--backend', str(backend),
                    '--report', str(selected_report), '--output', str(destination)], cwd=backend, check=True)
    print('Open START_HERE.md. Individual prompts and answers are in latest-baseline.jsonl and latest-pruning-predictions.jsonl.')
    print('Downloading:', destination.name)
    files.download(str(destination))
